Download dataset

In [62]:
import kagglehub
import pandas

path = kagglehub.dataset_download("xseeker0/translation-from-classi"
                                  "cal-chinese-to-english", output_dir="./data")

Read Dataset

In [63]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(path, "weyan_idioms.csv"))
df.head()  # Show first five rows

,zh,en
0,乾统三年，徙封秦国。,"In the third year of the Qingtong Era, he was ..."
1,莽曰监邛。,It was called Jianggongsu in Wang Mang's time.
2,辛未，以复置尚书省诏天下。,"On the tenth day of the first lunar month, an ..."
3,《司马法》广陈三代，曰：古者六尺为步，步百为亩，亩百为夫，夫三为屋，屋三为井。,The Book of Sima's Military Methods extensivel...
4,事平，加特进、右骁卫大将军，弘志右卫上将军兼中尉，守义右领军卫上将军。,"After the incident had ended, Qiu Shiliang was..."


#### Tokenize

|   |English| Chinese  |
|---|:------|:----|
| 1 |Convert to lowercase| \   |
| 2 |Add spaces before punctuation marks| \  |
|3|Seperate by space| Seperate by character |

In [64]:
import re


def tokenize_en(text: str) -> list[str]:
    basic = r"\S\w*"  # basic case
    hyphen = r"\w+(?:[-']\w+)*"  # handle hyphen: "pre-train"
    point = r"(?:\w+\.)+\w+(?:\.)*"  # handle point: "U.S.A"
    pattern = point + r"|" + hyphen + r"|" + basic
    return re.findall(pattern, text.lower())


def tokenize_zh(text: str) -> list[str]:
    return [char for char in text if not char.isspace()]


# Replace NaN with empty string
df['en'] = df['en'].fillna("")
df['zh'] = df['zh'].fillna("")

# Tokenize
df['en_tokens'] = df['en'].apply(tokenize_en)
df['zh_tokens'] = df['zh'].apply(tokenize_zh)

Tokenize result

In [65]:
df[['zh', 'zh_tokens']].head()

,zh,zh_tokens
0,乾统三年，徙封秦国。,"[乾, 统, 三, 年, ，, 徙, 封, 秦, 国, 。]"
1,莽曰监邛。,"[莽, 曰, 监, 邛, 。]"
2,辛未，以复置尚书省诏天下。,"[辛, 未, ，, 以, 复, 置, 尚, 书, 省, 诏, 天, 下, 。]"
3,《司马法》广陈三代，曰：古者六尺为步，步百为亩，亩百为夫，夫三为屋，屋三为井。,"[《, 司, 马, 法, 》, 广, 陈, 三, 代, ，, 曰, ：, 古, 者, 六, ..."
4,事平，加特进、右骁卫大将军，弘志右卫上将军兼中尉，守义右领军卫上将军。,"[事, 平, ，, 加, 特, 进, 、, 右, 骁, 卫, 大, 将, 军, ，, 弘, ..."


In [66]:
df[['en', 'en_tokens']].head()

,en,en_tokens
0,"In the third year of the Qingtong Era, he was ...","[in, the, third, year, of, the, qingtong, era,..."
1,It was called Jianggongsu in Wang Mang's time.,"[it, was, called, jianggongsu, in, wang, mang'..."
2,"On the tenth day of the first lunar month, an ...","[on, the, tenth, day, of, the, first, lunar, m..."
3,The Book of Sima's Military Methods extensivel...,"[the, book, of, sima's, military, methods, ext..."
4,"After the incident had ended, Qiu Shiliang was...","[after, the, incident, had, ended, ,, qiu, shi..."


#### Vocabulary statistics

In [71]:
from collections import Counter

# get all words
words_en = [word for tokens in df['en_tokens'] for word in tokens]
words_zh = [word for tokens in df['zh_tokens'] for word in tokens]

# Special tokens
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"


# Use Counter to get vocabulary
def get_vocabulary(words: list[str]) -> Counter:
    return (Counter([PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]  # Add pecial tokens
                    + words))


vocabulary_en = get_vocabulary(words_en)
vocabulary_zh = get_vocabulary(words_zh)

Most frequent tokens:


NameError: name 'pandas' is not defined

Result

In [80]:
print('Most frequent tokens:')
print(pd.concat([
    pd.DataFrame(vocabulary_en.most_common(10), columns=['Token', 'Count']),
    pd.DataFrame(vocabulary_zh.most_common(10), columns=['Token', 'Count'])],
    axis=1))

Most frequent tokens:
  Token    Count Token    Count
0   the  2263560     ，  2044683
1     ,  1982771     。   986391
2     .  1297585     之   418786
3   and  1246982     以   241976
4    of   953889     不   234412
5    to   906109     、   207778
6    in   446140     为   205824
7   was   387497     其   163281
8     a   377996     人   153195
9    he   325924     而   150151


#### Transformer Model
1. Multihead attention